# ERCOT Load Forecasting

## 03 — Weather Features

Notebook 02 ended with a hypothesis: the baseline models fail during rapid weather
changes — their ten worst errors were all ~22–26 GW underpredictions during the
January/February 2025 cold snaps — because lagged demand cannot anticipate a cold front.

This notebook adds hourly weather features and tests that hypothesis. Same methodology
as notebook 02 (chronological splits, TimeSeriesSplit, same baselines, focus on
Gradient Boosting and XGBoost) — the only thing that changes is the feature set.

In [1]:
import os
import requests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Weather data source

We use the **Open-Meteo historical weather API** (https://open-meteo.com/en/docs/historical-weather-api):
free, no API key (like the ERCOT download in notebook 01), hourly resolution, based on the
ERA5 reanalysis dataset. Variables: temperature, relative humidity, and apparent
("feels-like") temperature, all in °C / %.

Two decisions worth noting:

- **Where to measure.** ERCOT load is statewide but weather is local, so we take the four
  largest metros — Houston, Dallas, San Antonio, Austin — where most of the demand is, and
  will average them into one statewide weather signal. Simple, and easy to defend.
- **Timezone.** We request timestamps in **UTC** and convert to `America/Chicago` in pandas,
  exactly like the load data in notebook 02. Asking an API for local clock time invites the
  same daylight-saving ambiguity we already fought once.

In [3]:
cities = {
    "houston": (29.76, -95.36),
    "dallas": (32.78, -96.80),
    "san_antonio": (29.42, -98.49),
    "austin": (30.27, -97.74),
}
years = [2022, 2023, 2024, 2025]

weather_path = "../data/raw/weather_4cities.csv"

if os.path.exists(weather_path):
    print("already downloaded, skipping")
else:
    pieces = []
    # one request per city per year — a single 4-year request is too heavy
    # for the archive API and times out
    for city, (lat, lon) in cities.items():
        for year in years:
            print(f"{city} {year}: downloading...")
            params = {
                "latitude": lat,
                "longitude": lon,
                "start_date": f"{year}-01-01",
                "end_date": f"{year}-12-31",
                "hourly": "temperature_2m,relative_humidity_2m,apparent_temperature",
                "timezone": "UTC",
            }
            response = requests.get(
                "https://archive-api.open-meteo.com/v1/archive",
                params=params,
                timeout=120,
            )
            # stop with a clear error if the API is down; just re-run later
            response.raise_for_status()

            # the "hourly" part of the response is a dict of equal-length lists,
            # which is exactly what the DataFrame constructor wants
            df_piece = pd.DataFrame(response.json()["hourly"])
            df_piece["city"] = city
            pieces.append(df_piece)

    weather_raw = pd.concat(pieces, ignore_index=True)
    weather_raw.to_csv(weather_path, index=False)
    print("saved", weather_path)

already downloaded, skipping


houston 2023: downloading...


houston 2024: downloading...


houston 2025: downloading...


dallas 2022: downloading...


...

austin 2022: downloading...


austin 2023: downloading...


austin 2024: downloading...


austin 2025: downloading...


saved ../data/raw/weather_4cities.csv


In [4]:
weather_raw = pd.read_csv(weather_path)

print(weather_raw.shape)
weather_raw.head()

(140256, 5)


,time,temperature_2m,relative_humidity_2m,apparent_temperature,city
0,2022-01-01T00:00,23.2,89,24.9,houston
1,2022-01-01T01:00,22.8,90,24.6,houston
2,2022-01-01T02:00,23.2,88,24.9,houston
3,2022-01-01T03:00,23.7,86,25.1,houston
4,2022-01-01T04:00,23.3,87,24.8,houston


## 2. First inspection

Before merging anything: does every city have the same number of hours, are there missing
values, and do the temperature ranges look physically plausible for Texas?

In [5]:
print("hours per city:")
print(weather_raw["city"].value_counts())

print("\nmissing values per column:")
print(weather_raw.isna().sum())

print("\nduplicate (city, time) pairs:", weather_raw.duplicated(subset=["city", "time"]).sum())

hours per city:
city
houston        35064
dallas         35064
san_antonio    35064
austin         35064
Name: count, dtype: int64

missing values per column:
time                    0
temperature_2m          0
relative_humidity_2m    0
apparent_temperature    0
city                    0
dtype: int64

duplicate (city, time) pairs: 0


In [5]:
# temperature range per city, in Celsius — sanity check against known Texas weather
weather_raw.groupby("city")["temperature_2m"].describe()

,count,mean,std,min,25%,50%,75%,max
city,,,,,,,,
austin,35064.0,21.519456,8.888687,-9.5,15.6,22.8,27.8,41.6
dallas,35064.0,20.181956,9.744675,-11.8,13.3,21.4,27.5,42.9
houston,35064.0,21.652926,7.830504,-8.8,16.8,23.1,27.1,41.7
san_antonio,35064.0,22.189134,8.529548,-8.3,16.6,23.4,28.1,41.8


In [7]:
weather_raw["time"] = pd.to_datetime(
    weather_raw["time"],
    utc=True
)

weather_raw["time"] = weather_raw["time"].dt.tz_convert("America/Chicago")

print(weather_raw["time"].head())
print(weather_raw["time"].dtype)

print(weather_raw["time"].min())
print(weather_raw["time"].max())

0   2021-12-31 18:00:00-06:00
1   2021-12-31 19:00:00-06:00
2   2021-12-31 20:00:00-06:00
3   2021-12-31 21:00:00-06:00
4   2021-12-31 22:00:00-06:00
Name: time, dtype: datetime64[us, America/Chicago]
datetime64[us, America/Chicago]
2021-12-31 18:00:00-06:00
2025-12-31 17:00:00-06:00


In [8]:
#IMPORT THE PROCESSED DATA FROM NOTEBOOK 02 
model_df = pd.read_csv(
    "../data/processed/model_data.csv",
    parse_dates=["timestamp"],
    index_col="timestamp"
)

model_df.index = pd.to_datetime(model_df.index, utc=True)
model_df.index = model_df.index.tz_convert("America/Chicago")

In [10]:
print(model_df.index.min())
print(model_df.index.max())
print(model_df.index.dtype)

2022-01-08 01:00:00-06:00
2026-01-01 00:00:00-06:00
datetime64[us, America/Chicago]


In [11]:
print(weather_raw["time"].min())
print(weather_raw["time"].max())

print(model_df.index.min())
print(model_df.index.max())

2021-12-31 18:00:00-06:00
2025-12-31 17:00:00-06:00
2022-01-08 01:00:00-06:00
2026-01-01 00:00:00-06:00


In [16]:
extra_pieces = []

for city, (lat, lon) in cities.items():

    print(f"{city}: downloading 2026-01-01")

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2026-01-01",
        "end_date": "2026-01-01",
        "hourly": "temperature_2m,relative_humidity_2m,apparent_temperature",
        "timezone": "UTC",
    }

    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=120,
    )

    response.raise_for_status()

    df_piece = pd.DataFrame(response.json()["hourly"])
    df_piece["city"] = city

    extra_pieces.append(df_piece)


extra_weather = pd.concat(extra_pieces,ignore_index=True)

extra_weather["time"] = pd.to_datetime(
    extra_weather["time"],
    utc=True
)

extra_weather["time"] = extra_weather["time"].dt.tz_convert(
    "America/Chicago"
)

houston: downloading 2026-01-01
dallas: downloading 2026-01-01
san_antonio: downloading 2026-01-01
austin: downloading 2026-01-01


In [15]:
print(extra_weather["time"].min())
print(extra_weather["time"].max())
print(extra_weather.shape)



2025-12-31 18:00:00-06:00
2026-01-01 17:00:00-06:00
(96, 5)


In [17]:
#LETS APPEND IT

weather_raw = pd.concat([weather_raw, extra_weather],
    ignore_index=True
)

In [20]:
print("duplicates:",weather_raw.duplicated(subset=["city", "time"]).sum())



duplicates: 0


In [19]:
print("Weather:")
print(weather_raw["time"].min())
print(weather_raw["time"].max())

print("\nERCOT:")
print(model_df.index.min())
print(model_df.index.max())

Weather:
2021-12-31 18:00:00-06:00
2026-01-01 17:00:00-06:00

ERCOT:
2022-01-08 01:00:00-06:00
2026-01-01 00:00:00-06:00


In [21]:
weather_raw.to_csv(weather_path,index=False)

In [22]:
#PIVOTING

weather_wide = weather_raw.pivot(
    index="time",
    columns="city",
    values=[
        "temperature_2m",
        "relative_humidity_2m",
        "apparent_temperature"
    ]
)

weather_wide.head()

temperature_2m                             \
city                              austin dallas houston san_antonio   
time                                                                  
2021-12-31 18:00:00-06:00           24.0   20.4    23.2        25.3   
2021-12-31 19:00:00-06:00           22.9   19.8    22.8        24.2   
2021-12-31 20:00:00-06:00           22.2   18.8    23.2        23.4   
2021-12-31 21:00:00-06:00           21.5   18.8    23.7        22.2   
2021-12-31 22:00:00-06:00           21.4   18.0    23.3        21.8   

                          relative_humidity_2m                             \
city                                    austin dallas houston san_antonio   
time                                                                        
2021-12-31 18:00:00-06:00                 76.0   85.0    89.0        70.0   
2021-12-31 19:00:00-06:00                 82.0   89.0    90.0        76.0   
2021-12-31 20:00:00-06:00                 84.0   91.0    88.0        80.0   
2021-12-31 21:00:00-06:00                 89.0   94.0    86.0        92.0   
2021-12-31 22:00:00-06:00                 95.0   98.0    87.0        96.0   

                          apparent_temperature                             
city                                    austin dallas houston san_antonio  
time                                                                       
2021-12-31 18:00:00-06:00                 24.9   21.7    24.9        26.8  
2021-12-31 19:00:00-06:00                 23.8   21.7    24.6        26.0  
2021-12-31 20:00:00-06:00                 22.9   20.6    24.9        24.8  
2021-12-31 21:00:00-06:00                 22.3   20.4    25.1        23.6  
2021-12-31 22:00:00-06:00                 22.3   20.5    24.8        23.9

In [23]:
weather_wide.columns

MultiIndex([(      'temperature_2m',      'austin'),
            (      'temperature_2m',      'dallas'),
            (      'temperature_2m',     'houston'),
            (      'temperature_2m', 'san_antonio'),
            ('relative_humidity_2m',      'austin'),
            ('relative_humidity_2m',      'dallas'),
            ('relative_humidity_2m',     'houston'),
            ('relative_humidity_2m', 'san_antonio'),
            ('apparent_temperature',      'austin'),
            ('apparent_temperature',      'dallas'),
            ('apparent_temperature',     'houston'),
            ('apparent_temperature', 'san_antonio')],
           names=[None, 'city'])

In [24]:
weather_wide.columns = [
    f"{variable}_{city}"
    for variable, city in weather_wide.columns
]

In [27]:
weather_wide.head()
print(weather_wide.columns)
print("===========")
print(weather_wide.index.is_unique)
print(weather_wide.isna().sum())

Index(['temperature_2m_austin', 'temperature_2m_dallas',
       'temperature_2m_houston', 'temperature_2m_san_antonio',
       'relative_humidity_2m_austin', 'relative_humidity_2m_dallas',
       'relative_humidity_2m_houston', 'relative_humidity_2m_san_antonio',
       'apparent_temperature_austin', 'apparent_temperature_dallas',
       'apparent_temperature_houston', 'apparent_temperature_san_antonio'],
      dtype='str')
True
temperature_2m_austin               0
temperature_2m_dallas               0
temperature_2m_houston              0
temperature_2m_san_antonio          0
relative_humidity_2m_austin         0
relative_humidity_2m_dallas         0
relative_humidity_2m_houston        0
relative_humidity_2m_san_antonio    0
apparent_temperature_austin         0
apparent_temperature_dallas         0
apparent_temperature_houston        0
apparent_temperature_san_antonio    0
dtype: int64


In [31]:
merged_df = model_df.join(weather_wide, how="inner")

#how=inner KEEP ONLY TIMESTAMPS APPEARING IN BOTH DATASETS.

print(model_df.shape)
print(weather_wide.shape)
print(merged_df.shape)

(34896, 18)
(35088, 12)
(34896, 30)


In [30]:
#FINALLY AGAIN SAVE THE DATAFRAME AS .csv

merged_df.to_csv(
    "../data/processed/model_data_weather.csv",
    index=True,
    index_label="timestamp"
)